# BMED365: Lab 5 - Computational Modeling with Local LLMs
## Introduction to Computational Neuroscience & Medical Physics

**Objective:**
In this notebook, we will use a local Large Language Model (**Qwen 2.5 72B**) running on the M4 Max to simulate complex biological systems. We treat the LLM as a "Senior Computational Colleague"—we ask it to formulate mathematical models (ODEs, SDEs) and write high-performance Python code, which we then validate and run.

**Topics Covered:**
1.  **Dynamical Systems:** The FitzHugh-Nagumo Model (Action Potentials).
2.  **Stochastic Processes:** Membrane Noise (Ornstein-Uhlenbeck).
3.  **Medical Physics:** MRI Relaxation (Bloch Equations).

**Environment:** `mlx-bio` (Conda)
**Hardware:** Apple M4 Max (Metal Performance Shaders)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import mlx.core as mx
from mlx_lm import load, generate

# Configure Plotting Style for the Course
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 140

print(f"✅ MLX Device: {mx.default_device()}")

In [ ]:
# 1. Load the Model (Cached in ~/.cache/huggingface/hub)
model_id = "mlx-community/Qwen2.5-72B-Instruct-4bit"
print(f"Loading {model_id} into Unified Memory...")

# We use the default tokenizer config
model, tokenizer = load(model_id)

# 2. Define the "Professor" Persona (System Prompt)
SYSTEM_PROMPT = """
You are an expert Professor of Computational Medicine and Biomedical Physics.
Your goal is to translate biological problems into precise, high-performance computational models.

**Methodology:**
1.  **Dynamical Systems:** Use Phase Plane analysis (Nullclines) for excitability.
2.  **Stochasticity:** Use Euler-Maruyama for SDEs (never standard ODE solvers).
3.  **MRI/Physics:** Use Vectorized Bloch Equations.

**Coding Standards:**
* Use `numpy` vectorization (avoid loops).
* Always define units and governing equations in LaTeX.
* Use `scipy.integrate` for deterministic systems.
"""

def ask_professor(query, max_tokens=2048):
    """
    Sends a query to the local LLM with the Professor persona.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query}
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    print(f"🧠 Professor is thinking about: '{query}'...")
    start = time.time()
    
    response = generate(
        model, 
        tokenizer, 
        prompt=prompt, 
        verbose=False, 
        max_tokens=max_tokens, 
        temp=0.3 # Low temp for mathematical precision
    )
    
    end = time.time()
    print(f"✅ Generated in {end - start:.2f}s")
    return response

print("✅ Model & Agent Ready.")

## Example 1: The Excitable Brain (Deterministic ODEs)

We start with a classic model of neuronal excitability: the **FitzHugh-Nagumo (FHN)** model. It is a 2D simplification of the Hodgkin-Huxley model, perfect for understanding "Phase Space" dynamics.

**The Prompt:**
We will ask the model to:
1.  Define the equations.
2.  Simulate a neuron firing.
3.  **Crucially:** Plot the "Nullclines" (where $dV/dt = 0$) to show the stability.

In [ ]:
# We ask the LLM to generate the code for us
query_fhn = """
I need to model a neuron using the FitzHugh-Nagumo equations.
1. Define the ODEs with parameters a=0.7, b=0.8, tau=12.5.
2. Solve it for 100ms with an injected current I_ext = 0.5.
3. Plot two subplots: 
   (a) Voltage vs Time 
   (b) The Phase Plane (V vs W) with Nullclines and the trajectory.
"""

response_fhn = ask_professor(query_fhn)
print(response_fhn)

In [ ]:
# ---------------------------------------------------------
# COPY/PASTE THE GENERATED CODE HERE TO RUN IT
# (Below is the Reference Implementation for the Notebook)
# ---------------------------------------------------------

def fhn_neuron(y, t, I_ext, a=0.7, b=0.8, tau=12.5):
    V, W = y
    # dV/dt = V - V^3/3 - W + I
    dVdt = V - (V**3)/3 - W + I_ext
    # dW/dt = (V + a - b*W) / tau
    dWdt = (V + a - b * W) / tau
    return [dVdt, dWdt]

# Parameters
t = np.linspace(0, 100, 1000)
y0 = [-1.0, 1.0] # Initial state (Resting)
I_ext = 0.5

# Integrate
sol = odeint(fhn_neuron, y0, t, args=(I_ext,))
V = sol[:, 0]
W = sol[:, 1]

# Plotting
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Time Series
ax[0].plot(t, V, label='Voltage (V)', color='#d62728')
ax[0].set_title("Neuron Action Potential (Time Domain)")
ax[0].set_xlabel("Time (ms)")
ax[0].set_ylabel("Membrane Potential (a.u.)")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# Phase Plane & Nullclines
v_grid = np.linspace(-2.5, 2.5, 100)
# V-nullcline: dV/dt = 0 -> W = V - V^3/3 + I
w_null_v = v_grid - (v_grid**3)/3 + I_ext
# W-nullcline: dW/dt = 0 -> W = (V + a)/b
w_null_w = (v_grid + 0.7) / 0.8

ax[1].plot(v_grid, w_null_v, '--', label='V-Nullcline', color='grey')
ax[1].plot(v_grid, w_null_w, '--', label='W-Nullcline', color='blue')
ax[1].plot(V, W, '-', label='Trajectory', color='black', linewidth=2)
ax[1].set_title("Phase Space Analysis")
ax[1].set_xlabel("Voltage (V)")
ax[1].set_ylabel("Recovery Variable (W)")
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Example 2: Stochastic Processes (Membrane Noise)

Biological systems are noisy. The standard `odeint` solver cannot handle noise (randomness) because the derivative $dW_t/dt$ is undefined. We must use **Stochastic Differential Equations (SDEs)**.

**The Prompt:**
We ask the model to simulate an **Ornstein-Uhlenbeck (OU)** process, often used to model background synaptic noise or membrane voltage fluctuations. Note that the System Prompt enforces the **Euler-Maruyama** method.

In [ ]:
query_sde = """
Simulate the membrane potential noise of a neuron using an Ornstein-Uhlenbeck process.
Equation: dV = theta * (mu - V) * dt + sigma * dW
Parameters: theta=0.1 (decay), mu=-65 (resting mV), sigma=2.0 (noise).
Time: 0 to 1000ms.
Use the Euler-Maruyama method explicitly.
"""

response_sde = ask_professor(query_sde)
print(response_sde)

In [ ]:
# ---------------------------------------------------------
# Reference Implementation (Euler-Maruyama)
# ---------------------------------------------------------

# Parameters
theta = 0.1   # Speed of reversion to mean
mu = -65.0    # Resting potential (mV)
sigma = 2.0   # Volatility (Noise intensity)
T = 1000      # Total time (ms)
dt = 0.1      # Time step
N = int(T / dt)

# Arrays
t = np.linspace(0, T, N)
V = np.zeros(N)
V[0] = -65.0 # Initial condition

# Euler-Maruyama Loop
# dW is sampled from Normal(0, sqrt(dt))
# We generate all random numbers at once for vectorization speed (M4 Max optimization)
dW = np.random.normal(0, np.sqrt(dt), N)

for i in range(1, N):
    drift = theta * (mu - V[i-1]) * dt
    diffusion = sigma * dW[i]
    V[i] = V[i-1] + drift + diffusion

# Visualization
plt.figure(figsize=(10, 4))
plt.plot(t, V, color='#1f77b4', lw=0.8, alpha=0.8)
plt.axhline(mu, color='red', linestyle='--', label=f'Mean ({mu} mV)')
plt.title(f"Membrane Noise (Ornstein-Uhlenbeck Process)\n$\sigma={sigma}, \\theta={theta}$")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane Potential (mV)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Example 3: Medical Physics (MRI T1 Relaxation)



[Image of MRI Scanner Physics]


Finally, we look at **Computational Medicine**. In MRI, protons precess in a magnetic field. The **Bloch Equations** describe how the magnetization vector $\vec{M} = (M_x, M_y, M_z)$ relaxes back to equilibrium.

**The Prompt:**
We need to verify if the model can handle the 3D vector math and loop-free coding required for efficient image simulation.

In [ ]:
# ---------------------------------------------------------
# Reference Implementation
# ---------------------------------------------------------

# Time vector
t = np.linspace(0, 5000, 500) # 0 to 5 seconds
M0 = 1.0 # Normalized initial magnetization

# T1 values for brain tissues (in ms)
tissues = {
    "Cerebrospinal Fluid (CSF)": 4000,
    "Gray Matter": 1300,
    "White Matter": 800
}

plt.figure(figsize=(10, 6))

for name, T1 in tissues.items():
    # Vectorized calculation (No loops over time t)
    Mz = M0 * (1 - np.exp(-t / T1))
    plt.plot(t, Mz, label=f"{name} (T1={T1}ms)", lw=2.5)

plt.axhline(0, color='black', lw=1)
plt.title("MRI Physics: T1 Relaxation Recovery Curves")
plt.xlabel("Time (TR) [ms]")
plt.ylabel("Longitudinal Magnetization ($M_z$)")
plt.legend()
plt.grid(True, which='both', linestyle='--', alpha=0.5)

# Highlight: This intersection shows why image contrast depends on timing
plt.annotate('Contrast Window', xy=(1000, 0.6), xytext=(1500, 0.4),
             arrowprops=dict(facecolor='black', shrink=0.05))

plt.show()